In [75]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [76]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [77]:
rng = np.random.default_rng(73512)

In [78]:
# load data from pickle file
raw_data = pd.read_pickle('train.pkl')
np.random.shuffle(raw_data)

In [79]:
classes = [int(raw_data[i][1]) for i in range(len(raw_data))]
data = [torch.tensor(raw_data[i][0]).float() for i in range(len(raw_data))]
print(pd.Series(classes).unique())
print(len(data))
print(data[0].shape)

[3 0 1 4 2]
2939
torch.Size([140])


In [80]:
from torch.utils.data import Dataset

class VariableLenDataset(Dataset):
    def __init__(self, in_data, target):
        self.data = [(x, y) for x, y in zip(in_data, target)]      

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        in_data, target = self.data[idx]
        return in_data, target

In [81]:
all_data = torch.concatenate([x.flatten() for x in data])

data_mean = all_data.mean()
data_std = all_data.std()
data = [(x - data_mean) / data_std for x in data]
print(data_mean, data_std)

tensor(73.2270) tensor(55.0416)


In [82]:
targets = torch.tensor(classes, dtype=torch.long)

train_indices = int(len(data) * 0.7)
train_set = VariableLenDataset(data[:train_indices], targets[:train_indices])
valid_set = VariableLenDataset(data[train_indices:], targets[train_indices:])

train_set[0]

(tensor([ 0.7044,  0.7226,  0.8498, -0.2403,  1.7945,  1.7945,  1.2131, -1.0579,
         -0.4947, -0.4947, -1.0579,  1.7945,  0.9224,  1.7945,  1.7945, -0.5310,
          0.9770, -0.7854, -0.5310, -0.2403, -0.2040,  0.9770, -0.5310, -0.5310,
         -0.5310, -0.5310, -0.5310, -0.5310, -0.2403, -0.7854, -0.5310,  0.9770,
          1.5402,  0.9770,  0.9770, -0.5310,  1.5402,  1.4130,  0.3592, -0.5310,
         -0.5310, -0.5310, -0.5310, -0.5310, -0.5310, -0.5310,  0.3592, -0.5310,
         -0.5310,  1.5402,  0.9770,  1.8490,  1.8490,  1.5402, -0.2403, -0.7308,
         -0.7490, -0.7490, -0.6037, -0.5310, -0.5310, -0.5310, -0.5128, -0.4947,
         -1.0579,  0.9224, -1.0579, -0.8035,  1.7945,  1.7945,  1.7945,  1.7945,
          1.7945,  1.7945,  1.7945,  1.7945, -0.8035, -0.8035,  1.2131, -1.1124,
         -1.1124, -1.1124, -1.1124, -1.1124, -1.1124, -1.1124, -0.8217, -0.8217,
         -0.4765, -0.5310, -0.5310, -0.5310, -0.5310, -1.0942,  1.7945,  1.2131,
         -1.0579, -0.5310,  

In [83]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

pad = 0

def pad_collate(batch, pad_value=0):
    xx, yy = zip(*batch)
    x_lens = [len(x) for x in xx]

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    yy = torch.tensor(yy, dtype=torch.long)

    return xx_pad, yy, x_lens

In [84]:
BATCH_SIZE = 32
HIDDEN_SIZE = 4
NUM_LAYERS = 2
BIDIRECTIONAL = False
DROPOUT = 0.2
LR = 0.001
TRAIN_EPOCHS = 2

In [85]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, collate_fn=pad_collate)

In [86]:
print(len(train_set[0][0]))
print(len(train_set[1][0]))

140
102


In [87]:
next(iter(train_loader))

(tensor([[-0.0041, -0.0041, -0.0041,  ...,  0.0000,  0.0000,  0.0000],
         [-1.3304, -1.3122, -0.9670,  ...,  0.0000,  0.0000,  0.0000],
         [-1.3304, -1.3304,  0.7408,  ...,  0.0000,  0.0000,  0.0000],
         ...,
         [ 1.6128, -0.4402, -0.6763,  ...,  0.0000,  0.0000,  0.0000],
         [-1.3486, -1.3486, -1.3486,  ...,  0.0000,  0.0000,  0.0000],
         [-1.3486, -1.3486, -1.3486,  ...,  0.0000,  0.0000,  0.0000]]),
 tensor([3, 0, 4, 0, 0, 3, 1, 0, 3, 3, 0, 1, 0, 0, 1, 2, 0, 4, 0, 0, 3, 3, 0, 0,
         3, 0, 0, 0, 0, 2, 0, 1]),
 [216,
  116,
  461,
  192,
  204,
  452,
  1236,
  100,
  232,
  405,
  192,
  332,
  228,
  216,
  546,
  326,
  231,
  292,
  1993,
  360,
  336,
  204,
  1869,
  192,
  260,
  396,
  64,
  84,
  180,
  335,
  68,
  1095])

In [88]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, out_size, bidirectional = False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        if bidirectional:
            self.bidirectional = 2
        else:
            self.bidirectional = 1
        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size, num_layers = num_layers, bidirectional=bidirectional, dropout=DROPOUT)
        self.fc = nn.Linear(hidden_size*self.bidirectional, out_size) # do poprawy jeśli chce się korzystać z czego innego
        
    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        state = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        return hidden, state
    
    def forward(self, x, x_len, hidden):
        packed = nn.utils.rnn.pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
        packed_out, (hn, cn) = self.lstm(packed, hidden)
        # print(f'{hn.shape=}')
        if self.bidirectional == 1:
            out = hn[-1]
        else:
            forward_last = hn[-2]
            backward_last = hn[-1]
            out = torch.cat((forward_last, backward_last), dim=1)
                            
        # print(f'{out.shape=}')
        return self.fc(out), (hn, cn)
    
model = LSTMClassifier(1, HIDDEN_SIZE, NUM_LAYERS, 5, bidirectional=BIDIRECTIONAL).to(device)
model

LSTMClassifier(
  (lstm): LSTM(1, 4, num_layers=2, dropout=0.2)
  (fc): Linear(in_features=4, out_features=5, bias=True)
)

In [89]:
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
loss_fun = nn.CrossEntropyLoss()

from tqdm import tqdm
# Training loop
for epoch in tqdm(range(TRAIN_EPOCHS)):
    for x, targets, x_len in train_loader:
        x = x.to(device).unsqueeze(2)
        targets = targets.to(device)
        hidden, state = model.init_hidden(x.size(0))
        hidden, state = hidden.to(device), state.to(device) 
        preds, _ = model(x, x_len, (hidden,state))
        loss = loss_fun(preds, targets)
        optimizer.zero_grad() 
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}, loss: {loss.item():.3}")

 50%|█████     | 1/2 [01:33<01:33, 93.16s/it]

Epoch: 0, loss: 1.29


100%|██████████| 2/2 [03:09<00:00, 94.63s/it]


In [90]:
model.eval()
preds= []
targets= []

with torch.no_grad():
    for x, target, x_len in valid_loader:
        x = x.to(device).unsqueeze(2)
        target = target.to(device)
        hidden, state = model.init_hidden(x.shape[0])
        hidden, state = hidden.to(device), state.to(device)
        pred, _ = model(x, x_len, (hidden, state))
        preds.append(pred.cpu())
        targets.append(target.cpu())
print(f"Accuracy: {(torch.argmax(torch.cat(preds),1).cpu()==torch.cat(targets)).sum().item()/len(torch.cat(targets)):.3}")

Accuracy: 0.57


In [91]:
print(torch.argmax(torch.cat(preds),1).cpu())

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [98]:
print((torch.argmax(torch.cat(preds),1).cpu() == 0).sum())
print((torch.argmax(torch.cat(preds),1).cpu() != 0).sum())

tensor(882)
tensor(0)
